54, Motor de Popa Yamaha Evo Dash 155HP, propulsão

# Importações

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

# Limpeza Dados

In [2]:
df_vendas = pd.read_csv('../../Datasets/vendas_2023_2024.csv')
df_vendas.head()

,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,15-09-2024
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


In [3]:
vendas_iso = pd.to_datetime(df_vendas['sale_date'], format='%Y-%m-%d', errors='coerce')
vendas_br = pd.to_datetime(df_vendas['sale_date'], format='%d-%m-%Y', errors='coerce')

df_vendas['sale_date'] = vendas_iso.fillna(vendas_br)
df_vendas[df_vendas['sale_date'].isna()]

,id,id_client,id_product,qtd,total,sale_date


In [4]:
df_vendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 9895 entries, 0 to 9894
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          9895 non-null   int64         
 1   id_client   9895 non-null   int64         
 2   id_product  9895 non-null   int64         
 3   qtd         9895 non-null   int64         
 4   total       9895 non-null   float64       
 5   sale_date   9895 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(4)
memory usage: 464.0 KB


# Previsão

In [5]:
# Verifiquei o id_product de Motor de Popa Yamaha Evo Dash 155HP manualmente
df_produto = df_vendas[df_vendas['id_product'] == 54]

vendas_diarias = df_produto.groupby('sale_date')['qtd'].sum().reset_index()
vendas_diarias.set_index('sale_date', inplace=True)
print(vendas_diarias.head())

            qtd
sale_date      
2023-01-10    3
2023-02-06   13
2023-02-27   15
2023-03-04   14
2023-03-15    4


In [6]:
calendario = pd.date_range(start='2023-01-01', end='2024-01-31')
print(calendario)

DatetimeIndex(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04',
               '2023-01-05', '2023-01-06', '2023-01-07', '2023-01-08',
               '2023-01-09', '2023-01-10',
               ...
               '2024-01-22', '2024-01-23', '2024-01-24', '2024-01-25',
               '2024-01-26', '2024-01-27', '2024-01-28', '2024-01-29',
               '2024-01-30', '2024-01-31'],
              dtype='datetime64[us]', length=396, freq='D')


In [7]:
vendas_diarias = vendas_diarias.reindex(calendario, fill_value=0)
vendas_diarias.head(8)

,qtd
2023-01-01,0
2023-01-02,0
2023-01-03,0
2023-01-04,0
2023-01-05,0
2023-01-06,0
2023-01-07,0
2023-01-08,0


In [8]:
vendas_diarias['previsao_7_dias'] = vendas_diarias['qtd'].rolling(window=7).mean().shift(1)
vendas_diarias.head(8)

,qtd,previsao_7_dias
2023-01-01,0,NaN
2023-01-02,0,NaN
2023-01-03,0,NaN
2023-01-04,0,NaN
2023-01-05,0,NaN
2023-01-06,0,NaN
2023-01-07,0,NaN
2023-01-08,0,0.0


In [9]:
df_teste = vendas_diarias.loc['2024-01-01':'2024-01-31'].copy()
df_teste.dropna(subset=['previsao_7_dias'], inplace=True)

y_true = df_teste['qtd']
y_pred = df_teste['previsao_7_dias']

mae = mean_absolute_error(y_true,y_pred)
print(f"MAE Janeiro 2024: {mae:.4f}")

MAE Janeiro 2024: 0.9954


In [10]:
vendas_diarias.loc['2023-12-25':'2024-01-07']

,qtd,previsao_7_dias
2023-12-25,0,1.285714
2023-12-26,0,1.285714
2023-12-27,0,1.285714
2023-12-28,0,0.000000
2023-12-29,0,0.000000
2023-12-30,0,0.000000
2023-12-31,0,0.000000
2024-01-01,0,0.000000
2024-01-02,0,0.000000
2024-01-03,0,0.000000


In [11]:
df_teste.head(7)

,qtd,previsao_7_dias
2024-01-01,0,0.0
2024-01-02,0,0.0
2024-01-03,0,0.0
2024-01-04,0,0.0
2024-01-05,0,0.0
2024-01-06,0,0.0
2024-01-07,0,0.0


# Conclusões

Previsão = 0
MAE = 0.9954
O baseline não é adequado, é um produto que vende de maneira muito intermitente, muitos dias entre cada venda,
além disso também não consegue observar periodos em que existe venda mais frequente no caso de sazionalidade
Olhar só a média é ineficiente porque só funciona se o futuro funcionar exatamente como o passado recente, o que é ineficiente e irrealista nesse caso.
O processo de montar o modelo foi primeiro agrupar por data as vendas do item, preencher as datas faltantes com 0 vendas e a partir disso calcular a media por dia com os ultimos 7 dias com shift para não considerar a propria data.
O shift(1) garante que não há data leakage (a previsão do dia T usa dados de T-7 a T-1), e filtrar inicalmente os dados para não ter datas invalidas.